# Finite-shot centered-fidelity verification
Monte Carlo check of the estimator-specific operator-norm confidence radius for the two-dimensional active metric.

In [ ]:
import numpy as np

# Exact active quantities for stress test I
G=np.diag([0.25,4.0])
Hact=np.diag([0.5,32.0])
Delta,W=1.0,4.0
gminus=0.25
kappaH=np.linalg.cond(Hact)
epscrit=gminus/2*((Delta/W)*kappaH-1)

# Exact fidelity around the optimum: F(d1,d2)=cos^2(d1/2) cos^2(2 d2)
def F(d1,d2):
    return np.cos(d1/2)**2*np.cos(2*d2)**2

h=0.1; N=100_000; delta=0.05; r=2
# Fully centered deterministic finite-difference metric
G_h=np.zeros((2,2))
G_h[0,0]=(2-F(h,0)-F(-h,0))/(2*h*h)
G_h[1,1]=(2-F(0,h)-F(0,-h))/(2*h*h)
G_h[0,1]=G_h[1,0]=(-F(h,h)+F(h,-h)+F(-h,h)-F(-h,-h))/(8*h*h)
beta_h=np.linalg.norm(G_h-G,2)
stat=(r/(2*h*h))*np.sqrt((1/N)*np.log(r*(r+1)/delta))
eps_bound=beta_h+stat
K=(W/Delta)*(1+2*eps_bound/gminus)
print('G_h=',G_h)
print('beta_h=',beta_h,'statistical term=',stat,'epsilon bound=',eps_bound)
print('epsilon_crit=',epscrit,'K=',K,'kappa_H=',kappaH)
assert eps_bound<epscrit and K<kappaH

# Bernoulli-shot metric estimator. Each displaced fidelity setting gets N shots.
settings=[(h,0),(-h,0),(0,h),(0,-h),(h,h),(h,-h),(-h,h),(-h,-h)]
probs=np.array([F(*s) for s in settings])
rng=np.random.default_rng(20260910)
errors=[]; kappas=[]
def invsqrt(A):
    w,V=np.linalg.eigh(A); return V@np.diag(1/np.sqrt(w))@V.T
for _ in range(5000):
    fh=rng.binomial(N,probs)/N
    Gh=np.zeros((2,2))
    Gh[0,0]=(2-fh[0]-fh[1])/(2*h*h)
    Gh[1,1]=(2-fh[2]-fh[3])/(2*h*h)
    Gh[0,1]=Gh[1,0]=(-fh[4]+fh[5]+fh[6]-fh[7])/(8*h*h)
    err=np.linalg.norm(Gh-G,2); errors.append(err)
    M=Gh+eps_bound*np.eye(2)
    S=invsqrt(M)@Hact@invsqrt(M)
    kappas.append(np.linalg.cond(S))
errors=np.array(errors); kappas=np.array(kappas)
print('median/95%/99%/max error=',np.median(errors),*np.quantile(errors,[.95,.99]),errors.max())
print('radius violations=',np.sum(errors>eps_bound),'of',len(errors))
print('max realized kappa=',kappas.max(),'theorem K=',K)
assert np.all(errors<=eps_bound)
assert np.all(kappas<K)
